<a href="https://colab.research.google.com/github/syedahijabzahra/DevelopersHub-AI-ML-Internship/blob/main/Health_Query_Chatbot/Health_Query_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install transformers torch sentencepiece -q

In [4]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

# ── Load Model ────────────────────────────────────────────────
print("Loading model... (first run may take 1-2 mins)")
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-large")
model     = T5ForConditionalGeneration.from_pretrained("google/flan-t5-large")
print("Model loaded ✅")

# ── System Persona (Prompt Engineering) ───────────────────────
PERSONA = (
    "You are HealthBot, a friendly and helpful medical assistant. "
    "Provide clear, accurate, and safe general health information in simple language. "
    "Never diagnose conditions, never prescribe medications, and never recommend specific dosages. "
    "If the question needs a doctor, say so clearly. "
    "Always be warm, supportive, and easy to understand. "
)

DISCLAIMER = (
    "\n\n⚠️ Disclaimer: This is general health information only, not medical advice. "
    "Please consult a qualified healthcare professional for personal medical concerns."
)

# ── Safety Filter ─────────────────────────────────────────────
BLOCKED_KEYWORDS = [
    "how to overdose", "overdose on", "kill myself", "suicide method",
    "lethal dose", "how to die", "end my life", "self harm instructions",
    "how to poison", "drug combination to", "how much to take to die"
]

EMERGENCY_RESPONSE = (
    "🚨 Your message contains sensitive content I'm not able to assist with.\n\n"
    "If you or someone else is in danger, please contact:\n"
    "  • Emergency Services : 115 (Pakistan) / 911 (USA)\n"
    "  • Umang Helpline     : 0317-4288665 (Pakistan)\n"
    "  • Crisis Text Line   : Text HOME to 741741 (USA)\n\n"
    "Please reach out to a qualified professional immediately. You are not alone. 💙"
)

def safety_check(query: str) -> bool:
    return not any(kw in query.lower() for kw in BLOCKED_KEYWORDS)

# ── Prompt Engineering ────────────────────────────────────────
def build_prompt(user_query: str) -> str:
    return (
        f"{PERSONA}\n\n"
        f"User question: {user_query}\n\n"
        f"Answer:"
    )

# ── Generate Response ─────────────────────────────────────────
def generate(prompt: str) -> str:
    inputs  = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    outputs = model.generate(**inputs, max_new_tokens=256)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# ── Chatbot Function ──────────────────────────────────────────
def health_chatbot(user_query: str) -> str:
    user_query = user_query.strip()

    if not user_query:
        return "Please enter a health question."
    if not safety_check(user_query):
        return EMERGENCY_RESPONSE

    response = generate(build_prompt(user_query))
    return response + DISCLAIMER

# ── Demo: Test with example queries ───────────────────────────
def run_demo():
    test_queries = [
        "What causes a sore throat?",
        "Is paracetamol safe for children?",
        "What are the early symptoms of diabetes?",
        "How can I lower my blood pressure naturally?",
        "How much water should I drink daily?",
        "How to overdose on medicine",
    ]

    print("=" * 65)
    print("        DEMO — Running example queries")
    print("=" * 65)

    for query in test_queries:
        print(f"\n🙋 User : {query}")
        print(f"🤖 Bot  : {health_chatbot(query)}")
        print("-" * 65)

# ── Interactive Chat Loop ─────────────────────────────────────
def run_chat():
    print("\n" + "=" * 65)
    print("   🏥  HealthBot — General Health Information Assistant")
    print("   ⚠️   Not a substitute for professional medical advice")
    print("=" * 65)
    print("\nExample questions:")
    print("  • What causes a sore throat?")
    print("  • Is paracetamol safe for children?")
    print("  • What are symptoms of high blood pressure?")
    print("  • How can I improve my sleep?")
    print("\nType 'quit' to exit.\n" + "-" * 65)

    while True:
        user_input = input("\n🙋 You  : ").strip()
        if not user_input:
            continue
        if user_input.lower() in ("quit", "exit", "q"):
            print("\n🤖 Bot  : Take care! Consult a doctor for personal health concerns. Goodbye! 💙")
            break
        print(f"\n🤖 Bot  : {health_chatbot(user_input)}")

# ── Entry Point ───────────────────────────────────────────────
run_demo()
run_chat()

Loading model... (first run may take 1-2 mins)


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded ✅
        DEMO — Running example queries

🙋 User : What causes a sore throat?
🤖 Bot  : A sore throat is caused by a virus.

⚠️ Disclaimer: This is general health information only, not medical advice. Please consult a qualified healthcare professional for personal medical concerns.
-----------------------------------------------------------------

🙋 User : Is paracetamol safe for children?
🤖 Bot  : Paracetamol is a non-steroidal anti-inflammatory drug. It is not recommended for children under the age of two.

⚠️ Disclaimer: This is general health information only, not medical advice. Please consult a qualified healthcare professional for personal medical concerns.
-----------------------------------------------------------------

🙋 User : What are the early symptoms of diabetes?
🤖 Bot  : Diabetes is a condition that affects the body's glucose levels. The first symptoms of diabetes are: : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : :